# Short-Volshock Three-Fold BC Slice Screen V2

This notebook tests whether the latest fold-local target manifest with wf_2020 short-volshock enrichment can support a three-fold short-volshock BC screen before any PPO or retained-actor promotion. It keeps SOXLBacktest as a Qlib-aligned reference baseline and records all promotion/retention gates as blocked unless later official walk-forward replay passes.

In [1]:

from __future__ import annotations

import json
import math
import os
import random
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    raise RuntimeError(f'Could not locate repo root from {Path.cwd()}')
os.chdir(ROOT)

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.config import load_profile

EXPERIMENT = 'short_volshock_three_fold_bc_slice_screen_v2'
OUT_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
MANIFEST_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / 'short_volshock_foldlocal_wf2020_target_manifest_v1' / 'fold_local_target_manifest.csv'
UPSTREAM_SUMMARY_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / 'short_volshock_foldlocal_wf2020_target_manifest_v1' / 'summary.json'
FRAME_PATH = ROOT / 'data' / 'soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'risk_budget_usage_v1'

OUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FOLDS = ('wf_2020_train', 'wf_2022_train', 'wf_2023_2024_train')
UNIVERSAL_REQUIRED_SLICES = {
    'release_gate_state': 20,
    'is_downtrend_state': 100,
    'drawdown_control_state': 20,
}
SHORT_REQUIRED_SLICES = {
    'short_volshock_state': 80,
    'short_volshock_target_state': 80,
}
SLICE_COLS = tuple(UNIVERSAL_REQUIRED_SLICES) + tuple(SHORT_REQUIRED_SLICES) + ('is_rebound_cluster',)
GLOBAL_MIN_ACCURACY = 0.70
FOLD_MIN_ACCURACY = 0.70
UNIVERSAL_SLICE_MIN_ACCURACY = 0.60
SHORT_SLICE_MIN_ACCURACY = 0.60
TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n')

print('root', ROOT)
print('out_dir', OUT_DIR)


root /Users/rziiiii/dev/workspace
out_dir /Users/rziiiii/dev/workspace/runtime/model_selection/card/short_volshock_three_fold_bc_slice_screen_v2


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Load Fold-Local Targets

In [2]:

manifest = pd.read_csv(MANIFEST_PATH)
ENRICHED_WF2020_TARGET_PATH = ROOT / 'runtime' / 'model_selection' / 'card' / 'wf2020_short_volshock_target_enrichment_v1' / 'wf2020_short_volshock_target_contract.csv'
manifest.loc[manifest['fold'].astype(str).eq('wf_2020_train'), 'target_path'] = str(ENRICHED_WF2020_TARGET_PATH.relative_to(ROOT))
manifest.loc[manifest['fold'].astype(str).eq('wf_2020_train'), 'source_label'] = 'wf2020_short_volshock_target_enrichment'
upstream_summary = json.loads(UPSTREAM_SUMMARY_PATH.read_text()) if UPSTREAM_SUMMARY_PATH.exists() else {}
frame = pd.read_pickle(FRAME_PATH)
profile = load_profile(PROFILE_NAME)

parts: list[pd.DataFrame] = []
for row in manifest.to_dict('records'):
    target_path = ROOT / str(row['target_path'])
    target = pd.read_csv(target_path)
    fold = str(row['fold'])
    if 'fold' in target.columns:
        target = target[target['fold'].astype(str).eq(fold)].copy()
    else:
        target['fold'] = fold
    if 'window' not in target.columns:
        target['window'] = fold
    target['target_fold'] = fold
    target['final_window'] = str(row.get('final_window') or fold.replace('_train', ''))
    target['source_label'] = str(row.get('source_label') or '')
    target['source_target_path'] = str(row['target_path'])
    parts.append(target)

contract = pd.concat(parts, ignore_index=True, sort=False)
contract['datetime'] = pd.to_datetime(contract['datetime'])
for col in (
    'is_rebound_cluster',
    'is_downtrend_state',
    'release_gate_state',
    'drawdown_control_state',
    'short_volshock_state',
    'short_volshock_target_state',
):
    if col not in contract.columns:
        contract[col] = False
    contract[col] = contract[col].fillna(False).infer_objects(copy=False).astype(bool)

fold_order = {fold: idx for idx, fold in enumerate(REQUIRED_FOLDS)}
contract['_fold_order'] = contract['fold'].map(fold_order).fillna(999).astype(int)
contract = contract.sort_values(['_fold_order', 'datetime', 'source_label']).drop(columns=['_fold_order']).reset_index(drop=True)
contract.to_csv(OUT_DIR / 'combined_target_contract.csv', index=False)
manifest.to_csv(OUT_DIR / 'source_manifest.csv', index=False)

fold_dataset_rows = []
for fold in REQUIRED_FOLDS:
    fold_targets = contract[contract['fold'].astype(str).eq(fold)].copy()
    if fold_targets.empty:
        fold_dataset_rows.append({
            'fold': fold,
            'source_label': '',
            'contract_rows': 0,
            'dataset_samples': 0,
            'start': '',
            'end': '',
            'release_gate_samples': 0,
            'downtrend_samples': 0,
            'drawdown_control_samples': 0,
            'short_volshock_state_samples': 0,
            'short_volshock_target_samples': 0,
        })
        continue
    dataset = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile)
    metadata = dataset.metadata
    def bsum(name: str) -> int:
        return int(metadata.get(name, pd.Series(False, index=metadata.index)).fillna(False).astype(bool).sum())
    fold_dataset_rows.append({
        'fold': fold,
        'source_label': ','.join(sorted(fold_targets['source_label'].dropna().astype(str).unique())),
        'contract_rows': int(len(fold_targets)),
        'dataset_samples': int(len(dataset.actions)),
        'dropped_rows_before_dataset': int(len(fold_targets) - len(dataset.actions)),
        'start': pd.Timestamp(fold_targets['datetime'].min()).date().isoformat(),
        'end': pd.Timestamp(fold_targets['datetime'].max()).date().isoformat(),
        'release_gate_samples': bsum('release_gate_state'),
        'downtrend_samples': bsum('is_downtrend_state'),
        'drawdown_control_samples': bsum('drawdown_control_state'),
        'short_volshock_state_samples': bsum('short_volshock_state'),
        'short_volshock_target_samples': bsum('short_volshock_target_state'),
    })

fold_dataset_audit = pd.DataFrame(fold_dataset_rows)
fold_dataset_audit.to_csv(OUT_DIR / 'fold_dataset_audit.csv', index=False)
missing_required_folds = [fold for fold in REQUIRED_FOLDS if fold not in set(contract['fold'].astype(str))]
short_feature_folds = fold_dataset_audit.loc[
    fold_dataset_audit['short_volshock_target_samples'].astype(int) >= min(SHORT_REQUIRED_SLICES.values()),
    'fold',
].astype(str).tolist()
short_feature_missing_folds = [fold for fold in REQUIRED_FOLDS if fold not in short_feature_folds]
required_fold_coverage = not missing_required_folds
short_volshock_feature_coverage_gate = len(short_feature_missing_folds) == 0

dataset_summary = {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'feature_frame': str(FRAME_PATH.relative_to(ROOT)),
    'source_manifest': str(MANIFEST_PATH.relative_to(ROOT)),
    'upstream_decision': upstream_summary.get('decision'),
    'upstream_training_allowed': bool(upstream_summary.get('training_allowed', False)),
    'combined_contract': 'combined_target_contract.csv',
    'contract_rows': int(len(contract)),
    'unique_dates': int(contract['datetime'].nunique()),
    'required_folds': list(REQUIRED_FOLDS),
    'missing_required_folds': missing_required_folds,
    'required_fold_coverage': bool(required_fold_coverage),
    'short_feature_folds': short_feature_folds,
    'short_feature_missing_folds': short_feature_missing_folds,
    'short_volshock_feature_coverage_gate': bool(short_volshock_feature_coverage_gate),
    'fold_dataset_audit': fold_dataset_audit.to_dict('records'),
}
write_json(OUT_DIR / 'dataset_summary.json', dataset_summary)
fold_dataset_audit


,fold,source_label,contract_rows,dataset_samples,dropped_rows_before_dataset,start,end,release_gate_samples,downtrend_samples,drawdown_control_samples,short_volshock_state_samples,short_volshock_target_samples
0,wf_2020_train,wf2020_short_volshock_target_enrichment,2470,2157,313,2010-03-11,2019-12-31,36,794,640,397,382
1,wf_2022_train,short_volshock_target_generation_handoff,253,253,0,2020-01-02,2020-12-31,40,152,66,135,133
2,wf_2023_2024_train,short_volshock_target_generation_handoff,756,755,1,2020-01-02,2022-12-30,40,507,65,341,335


## BC Weight And Slice Screen

In [3]:

TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)
LONG_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=600, lr=0.001, batch_size=128, seed=SEED)
WEIGHT_CONFIGS: list[dict[str, Any]] = [
    {
        'config_name': 'plain',
        'complexity': 0,
        'train_config': TRAIN_CONFIG,
        'sample_weight_config': None,
        'description': 'No sample weighting; verifies raw fold-local target learnability.',
    },
    {
        'config_name': 'downtrend2_obj15_cb99',
        'complexity': 2,
        'train_config': TRAIN_CONFIG,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=3.0,
            downtrend_multiplier=2.0,
            objective_weight_power=0.15,
            class_balanced_beta=0.99,
            weight_cap=30.0,
        ),
        'description': 'Moderate downtrend and objective-weight emphasis for short-volshock cap rows.',
    },
    {
        'config_name': 'release10_downtrend3_obj25_cb99',
        'complexity': 3,
        'train_config': TRAIN_CONFIG,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=3.0,
            release_gate_multiplier=10.0,
            objective_weight_power=0.25,
            class_balanced_beta=0.99,
            weight_cap=35.0,
        ),
        'description': 'Short-volshock oriented sparse-state weighting with class balance.',
    },
    {
        'config_name': 'release10_downtrend3_obj25_cb99_steps600',
        'complexity': 4,
        'train_config': LONG_TRAIN_CONFIG,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=3.0,
            release_gate_multiplier=10.0,
            objective_weight_power=0.25,
            class_balanced_beta=0.99,
            weight_cap=35.0,
        ),
        'description': 'Same short-volshock weighting with a longer BC schedule before any PPO.',
    },
]


def _metric_loss(value: float, threshold: float) -> float:
    if value is None or not math.isfinite(float(value)):
        return 1.0
    return max(0.0, float(threshold) - float(value))


def _sample_loss(samples: int, min_samples: int) -> float:
    return 0.0 if int(samples) >= int(min_samples) else (int(min_samples) - int(samples)) / max(int(min_samples), 1)

weight_config_rows = []
screen_rows: list[dict[str, Any]] = []
slice_rows: list[dict[str, Any]] = []
fold_rows: list[dict[str, Any]] = []
action_rows: list[dict[str, Any]] = []

for item in WEIGHT_CONFIGS:
    config_name = item['config_name']
    complexity = int(item['complexity'])
    train_config = item['train_config']
    sample_weight_config = item['sample_weight_config']
    weight_config_rows.append({
        'config_name': config_name,
        'complexity': complexity,
        'train_steps': int(train_config.steps),
        'lr': float(train_config.lr),
        'batch_size': int(train_config.batch_size),
        'sample_weight_config': json.dumps(json_safe(sample_weight_config.__dict__ if sample_weight_config else {}), sort_keys=True),
        'description': item['description'],
    })

    config_loss = 0.0
    per_fold_accuracy: list[float] = []
    per_fold_samples: list[int] = []
    per_slice_values: dict[str, list[dict[str, Any]]] = {name: [] for name in [*UNIVERSAL_REQUIRED_SLICES, *SHORT_REQUIRED_SLICES]}

    for fold in REQUIRED_FOLDS:
        fold_targets = contract[contract['fold'].astype(str).eq(fold)].copy()
        if fold_targets.empty:
            config_loss += 1.0
            fold_rows.append({
                'config_name': config_name,
                'complexity': complexity,
                'fold': fold,
                'samples': 0,
                'loss': math.nan,
                'accuracy': math.nan,
                'fold_gate': False,
                'min_accuracy': FOLD_MIN_ACCURACY,
                'before_accuracy': math.nan,
                'after_accuracy': math.nan,
                'after_loss': math.nan,
            })
            continue

        torch.manual_seed(SEED)
        np.random.seed(SEED)
        random.seed(SEED)
        dataset = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile)
        fit = fit_state_objective_bc_actor_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            train_config=train_config,
            sample_weight_config=sample_weight_config,
        )
        fold_eval = evaluate_bc_actor(fit.actor, dataset.obs, dataset.actions)
        fold_accuracy = float(fold_eval['accuracy'])
        fold_samples = int(fold_eval['samples'])
        per_fold_accuracy.append(fold_accuracy)
        per_fold_samples.append(fold_samples)
        fold_pass = fold_accuracy >= FOLD_MIN_ACCURACY
        fold_rows.append({
            'config_name': config_name,
            'complexity': complexity,
            'fold': fold,
            'samples': fold_samples,
            'loss': float(fold_eval['loss']),
            'accuracy': fold_accuracy,
            'fold_gate': bool(fold_pass),
            'min_accuracy': FOLD_MIN_ACCURACY,
            'before_accuracy': float(fit.metrics['before_accuracy']),
            'after_accuracy': float(fit.metrics['after_accuracy']),
            'after_loss': float(fit.metrics['after_loss']),
            'short_volshock_target_samples': int(dataset.metadata.get('short_volshock_target_state', pd.Series(False, index=dataset.metadata.index)).fillna(False).astype(bool).sum()),
        })
        config_loss += _metric_loss(fold_accuracy, FOLD_MIN_ACCURACY)

        metrics = evaluate_bc_actor_slice_fit(
            fit.actor,
            dataset.obs,
            dataset.actions,
            dataset.metadata,
            slice_cols=SLICE_COLS,
        )
        for row in metrics.to_dict('records'):
            row = dict(row)
            row.update({'config_name': config_name, 'complexity': complexity, 'fold': fold})
            slice_name = str(row['slice'])
            if slice_name in UNIVERSAL_REQUIRED_SLICES:
                min_samples = UNIVERSAL_REQUIRED_SLICES[slice_name]
                min_accuracy = UNIVERSAL_SLICE_MIN_ACCURACY
                required_scope = 'universal'
                sample_gate = int(row['samples']) >= int(min_samples)
            elif slice_name in SHORT_REQUIRED_SLICES:
                min_samples = SHORT_REQUIRED_SLICES[slice_name]
                min_accuracy = SHORT_SLICE_MIN_ACCURACY
                required_scope = 'short_volshock'
                sample_gate = fold in short_feature_folds and int(row['samples']) >= int(min_samples)
            else:
                min_samples = 0
                min_accuracy = math.nan
                required_scope = 'diagnostic'
                sample_gate = True
            accuracy = float(row['accuracy']) if pd.notna(row['accuracy']) else math.nan
            accuracy_gate = (slice_name not in [*UNIVERSAL_REQUIRED_SLICES, *SHORT_REQUIRED_SLICES]) or (
                sample_gate and math.isfinite(accuracy) and accuracy >= min_accuracy
            )
            row['required_scope'] = required_scope
            row['min_samples'] = int(min_samples)
            row['min_accuracy'] = min_accuracy if math.isfinite(float(min_accuracy)) else None
            row['sample_gate'] = bool(sample_gate)
            row['slice_gate'] = bool(accuracy_gate)
            slice_rows.append(row)
            if slice_name in per_slice_values:
                per_slice_values[slice_name].append({
                    'fold': fold,
                    'samples': int(row['samples']),
                    'accuracy': accuracy,
                    'slice_gate': bool(accuracy_gate),
                    'sample_gate': bool(sample_gate),
                })
                if slice_name in UNIVERSAL_REQUIRED_SLICES:
                    config_loss += _sample_loss(int(row['samples']), UNIVERSAL_REQUIRED_SLICES[slice_name])
                    config_loss += _metric_loss(accuracy, UNIVERSAL_SLICE_MIN_ACCURACY)
                elif slice_name in SHORT_REQUIRED_SLICES and fold in short_feature_folds:
                    config_loss += _sample_loss(int(row['samples']), SHORT_REQUIRED_SLICES[slice_name])
                    config_loss += _metric_loss(accuracy, SHORT_SLICE_MIN_ACCURACY)

        action_dist = fit.action_distribution.copy()
        action_dist['config_name'] = config_name
        action_dist['complexity'] = complexity
        action_dist['fold'] = fold
        action_rows.extend(action_dist.to_dict('records'))

    total_samples = int(sum(per_fold_samples))
    weighted_accuracy = float(np.average(per_fold_accuracy, weights=per_fold_samples)) if total_samples else math.nan
    min_fold_accuracy = float(min(per_fold_accuracy)) if per_fold_accuracy else math.nan
    global_accuracy_gate = math.isfinite(weighted_accuracy) and weighted_accuracy >= GLOBAL_MIN_ACCURACY
    fold_accuracy_gate = math.isfinite(min_fold_accuracy) and min_fold_accuracy >= FOLD_MIN_ACCURACY

    screen_row = {
        'config_name': config_name,
        'complexity': complexity,
        'train_steps': int(train_config.steps),
        'lr': float(train_config.lr),
        'batch_size': int(train_config.batch_size),
        'description': item['description'],
        'folds': len(per_fold_accuracy),
        'required_folds': len(REQUIRED_FOLDS),
        'missing_required_folds': ','.join(missing_required_folds),
        'required_fold_coverage': bool(required_fold_coverage),
        'short_feature_folds': ','.join(short_feature_folds),
        'short_feature_missing_folds': ','.join(short_feature_missing_folds),
        'short_volshock_feature_coverage_gate': bool(short_volshock_feature_coverage_gate),
        'total_samples': total_samples,
        'weighted_accuracy': weighted_accuracy,
        'min_fold_accuracy': min_fold_accuracy,
        'global_accuracy_gate': bool(global_accuracy_gate),
        'fold_accuracy_gate': bool(fold_accuracy_gate),
    }
    universal_slice_gate = True
    for name, min_samples in UNIVERSAL_REQUIRED_SLICES.items():
        vals = per_slice_values[name]
        min_samples_seen = min((int(v['samples']) for v in vals), default=0)
        min_accuracy_seen = min((float(v['accuracy']) for v in vals if math.isfinite(float(v['accuracy']))), default=math.nan)
        gate = bool(vals and min_samples_seen >= int(min_samples) and math.isfinite(min_accuracy_seen) and min_accuracy_seen >= UNIVERSAL_SLICE_MIN_ACCURACY)
        universal_slice_gate = universal_slice_gate and gate
        screen_row[f'{name}_min_samples'] = min_samples_seen
        screen_row[f'{name}_min_accuracy'] = min_accuracy_seen
        screen_row[f'{name}_gate'] = bool(gate)

    short_slice_gate = True
    for name, min_samples in SHORT_REQUIRED_SLICES.items():
        vals = [v for v in per_slice_values[name] if v['fold'] in short_feature_folds]
        min_samples_seen = min((int(v['samples']) for v in vals), default=0)
        min_accuracy_seen = min((float(v['accuracy']) for v in vals if math.isfinite(float(v['accuracy']))), default=math.nan)
        gate = bool(vals and min_samples_seen >= int(min_samples) and math.isfinite(min_accuracy_seen) and min_accuracy_seen >= SHORT_SLICE_MIN_ACCURACY)
        short_slice_gate = short_slice_gate and gate
        screen_row[f'{name}_eligible_fold_min_samples'] = min_samples_seen
        screen_row[f'{name}_eligible_fold_min_accuracy'] = min_accuracy_seen
        screen_row[f'{name}_eligible_gate'] = bool(gate)

    bc_screen_gate = bool(required_fold_coverage and global_accuracy_gate and fold_accuracy_gate and universal_slice_gate and short_slice_gate)
    ppo_screen_allowed_next = bool(bc_screen_gate and short_volshock_feature_coverage_gate)
    screen_row.update({
        'universal_slice_gate': bool(universal_slice_gate),
        'short_slice_gate': bool(short_slice_gate),
        'bc_screen_gate': bool(bc_screen_gate),
        'ppo_screen_allowed_next': bool(ppo_screen_allowed_next),
        'checkpoint_screen_allowed_next': bool(bc_screen_gate),
        'training_allowed_from_this_card': False,
        'promotion_allowed': False,
        'actor_retention_allowed': False,
        'gate_loss': float(config_loss + (0.5 if not short_volshock_feature_coverage_gate else 0.0)),
    })
    screen_rows.append(screen_row)

bc_weight_configs = pd.DataFrame(weight_config_rows)
bc_weight_screen = pd.DataFrame(screen_rows).sort_values(['bc_screen_gate', 'gate_loss', 'complexity'], ascending=[False, True, True]).reset_index(drop=True)
fold_bc_metrics = pd.DataFrame(fold_rows)
state_slice_bc_metrics = pd.DataFrame(slice_rows)
bc_action_distribution = pd.DataFrame(action_rows)

bc_weight_configs.to_csv(OUT_DIR / 'bc_weight_configs.csv', index=False)
bc_weight_screen.to_csv(OUT_DIR / 'bc_weight_screen.csv', index=False)
fold_bc_metrics.to_csv(OUT_DIR / 'fold_bc_metrics.csv', index=False)
state_slice_bc_metrics.to_csv(OUT_DIR / 'state_slice_bc_metrics.csv', index=False)
bc_action_distribution.to_csv(OUT_DIR / 'bc_action_distribution.csv', index=False)

bc_weight_screen


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


,config_name,complexity,train_steps,lr,batch_size,description,folds,required_folds,missing_required_folds,required_fold_coverage,...,short_volshock_target_state_eligible_gate,universal_slice_gate,short_slice_gate,bc_screen_gate,ppo_screen_allowed_next,checkpoint_screen_allowed_next,training_allowed_from_this_card,promotion_allowed,actor_retention_allowed,gate_loss
0,plain,0,300,0.001,128,No sample weighting; verifies raw fold-local t...,3,3,,True,...,True,False,True,False,False,False,False,False,False,0.127778
1,release10_downtrend3_obj25_cb99_steps600,4,600,0.001,128,Same short-volshock weighting with a longer BC...,3,3,,True,...,True,False,True,False,False,False,False,False,False,0.151916
2,release10_downtrend3_obj25_cb99,3,300,0.001,128,Short-volshock oriented sparse-state weighting...,3,3,,True,...,True,False,True,False,False,False,False,False,False,0.202411
3,downtrend2_obj15_cb99,2,300,0.001,128,Moderate downtrend and objective-weight emphas...,3,3,,True,...,True,False,True,False,False,False,False,False,False,0.399173


## Selection And Conclusion

In [4]:

passed = bc_weight_screen[bc_weight_screen['bc_screen_gate'].astype(bool)]
if not passed.empty:
    selected = passed.sort_values(['ppo_screen_allowed_next', 'gate_loss', 'complexity'], ascending=[False, True, True]).iloc[0].to_dict()
else:
    selected = bc_weight_screen.sort_values(['gate_loss', 'complexity']).iloc[0].to_dict()

selected_config = str(selected['config_name'])
selected_folds = fold_bc_metrics[fold_bc_metrics['config_name'].eq(selected_config)].copy()
selected_slices = state_slice_bc_metrics[state_slice_bc_metrics['config_name'].eq(selected_config)].copy()

failed_checks: list[str] = []
if not bool(selected.get('required_fold_coverage')):
    failed_checks.append('required_fold_coverage')
if not bool(selected.get('global_accuracy_gate')):
    failed_checks.append('global_accuracy')
if not bool(selected.get('fold_accuracy_gate')):
    failed_checks.append('fold_accuracy')
if not bool(selected.get('universal_slice_gate')):
    failed_checks.append('universal_slice_gate')
if not bool(selected.get('short_slice_gate')):
    failed_checks.append('short_volshock_slice_fit')
if not bool(selected.get('short_volshock_feature_coverage_gate')):
    failed_checks.append('short_volshock_feature_coverage_all_required_folds')

if bool(selected.get('ppo_screen_allowed_next')):
    decision = 'short_volshock_three_fold_bc_slice_ready_for_ppo_screen_no_promotion'
    next_mechanism = 'run_short_volshock_walkforward_checkpoint_or_ppo_screen_with_official_replay_gate'
elif bool(selected.get('bc_screen_gate')):
    decision = 'short_volshock_bc_slice_fit_ready_but_feature_coverage_blocks_ppo'
    next_mechanism = 'generate_wf2020_short_volshock_feature_targets_or_route_to_checkpoint_screen_reference_only'
else:
    decision = 'short_volshock_three_fold_bc_slice_failed_no_ppo'
    next_mechanism = 'retune_short_volshock_targets_or_weighting_before_any_ppo'

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'selected_config': selected_config,
    'selected_gate_loss': float(selected['gate_loss']),
    'selected_weighted_accuracy': float(selected['weighted_accuracy']),
    'selected_min_fold_accuracy': float(selected['min_fold_accuracy']),
    'bc_screen_gate': bool(selected['bc_screen_gate']),
    'short_volshock_feature_coverage_gate': bool(selected['short_volshock_feature_coverage_gate']),
    'short_feature_folds': short_feature_folds,
    'short_feature_missing_folds': short_feature_missing_folds,
    'ppo_screen_allowed_next': bool(selected['ppo_screen_allowed_next']),
    'checkpoint_screen_allowed_next': bool(selected['checkpoint_screen_allowed_next']),
    'training_allowed_from_this_card': False,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'failed_checks': failed_checks,
    'next_mechanism': next_mechanism,
    'required_folds': list(REQUIRED_FOLDS),
    'missing_required_folds': missing_required_folds,
    'universal_required_slices': UNIVERSAL_REQUIRED_SLICES,
    'short_required_slices': SHORT_REQUIRED_SLICES,
    'global_min_accuracy': GLOBAL_MIN_ACCURACY,
    'fold_min_accuracy': FOLD_MIN_ACCURACY,
    'universal_slice_min_accuracy': UNIVERSAL_SLICE_MIN_ACCURACY,
    'short_slice_min_accuracy': SHORT_SLICE_MIN_ACCURACY,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'source_manifest': str(MANIFEST_PATH.relative_to(ROOT)),
    'source_upstream_decision': upstream_summary.get('decision'),
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'short_volshock_foldlocal_wf2020_target_manifest_v1',
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference',
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'training_objective'],
    'promotion_blocker': 'BC slice fit is not official replay; promotion still requires retained actor replay across wf_2020/wf_2022/wf_2023_2024, MaxDD < 20%, and YTD return alignment.',
    'actor_retention': 'no actor retained by this BC learnability screen',
    'sidecars': [
        'source_manifest.csv',
        'combined_target_contract.csv',
        'dataset_summary.json',
        'fold_dataset_audit.csv',
        'bc_weight_configs.csv',
        'bc_weight_screen.csv',
        'fold_bc_metrics.csv',
        'state_slice_bc_metrics.csv',
        'bc_action_distribution.csv',
        'bc_slice_summary.json',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
}

bc_slice_summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'selected_config': selected_config,
    'selected': selected,
    'failed_checks': failed_checks,
    'selected_fold_rows': selected_folds.to_dict('records'),
    'selected_required_slice_rows': selected_slices[selected_slices['slice'].isin([*UNIVERSAL_REQUIRED_SLICES, *SHORT_REQUIRED_SLICES])].to_dict('records'),
}
summary = dict(selection)
summary['screen_rows'] = bc_weight_screen.to_dict('records')
summary['fold_dataset_audit'] = fold_dataset_audit.to_dict('records')

write_json(OUT_DIR / 'bc_slice_summary.json', bc_slice_summary)
write_json(OUT_DIR / 'selection.json', selection)
write_json(OUT_DIR / 'summary.json', summary)

fold_lines = []
for row in selected_folds.itertuples(index=False):
    fold_lines.append(
        f"- {row.fold}: accuracy {row.accuracy:.3f}, samples {int(row.samples)}, "
        f"short_target_samples {int(row.short_volshock_target_samples)}, gate {bool(row.fold_gate)}"
    )

slice_lines = []
required_rows = selected_slices[selected_slices['slice'].isin([*UNIVERSAL_REQUIRED_SLICES, *SHORT_REQUIRED_SLICES])]
for (slice_name, scope), rows in required_rows.groupby(['slice', 'required_scope']):
    eligible = rows if scope != 'short_volshock' else rows[rows['fold'].isin(short_feature_folds)]
    if eligible.empty:
        slice_lines.append(f"- {slice_name} ({scope}): no eligible rows, gate False")
        continue
    slice_lines.append(
        f"- {slice_name} ({scope}): min accuracy {eligible['accuracy'].min():.3f}, "
        f"min samples {int(eligible['samples'].min())}, gate {bool(eligible['slice_gate'].all())}"
    )

conclusion = f"""
## Conclusion

Decision: `{decision}`.

Selected BC config: `{selected_config}` with weighted fold accuracy `{float(selected['weighted_accuracy']):.3f}`, min fold accuracy `{float(selected['min_fold_accuracy']):.3f}`, and gate loss `{float(selected['gate_loss']):.4f}`.

Three-fold coverage: required folds `{', '.join(REQUIRED_FOLDS)}`; missing `{', '.join(missing_required_folds) if missing_required_folds else 'none'}`. Short-volshock feature folds: `{', '.join(short_feature_folds) if short_feature_folds else 'none'}`; missing `{', '.join(short_feature_missing_folds) if short_feature_missing_folds else 'none'}`.

Fold fit:
""" + "\n".join(fold_lines) + "\n\nRequired slice fit:\n" + "\n".join(slice_lines) + f"""

Gate note: `bc_screen_gate={bool(selection['bc_screen_gate'])}`, but `ppo_screen_allowed_next={bool(selection['ppo_screen_allowed_next'])}` because full short-volshock feature coverage across all required folds is `{bool(selection['short_volshock_feature_coverage_gate'])}`. `promotion_allowed=false` and `actor_retention_allowed=false` because this is only a BC learnability screen; retained-model promotion still requires official replay, MaxDD < 20%, and return alignment against current best and SOXLBacktest baseline.

Next action: `{next_mechanism}`.

Artifacts: `combined_target_contract.csv`, `fold_dataset_audit.csv`, `bc_weight_screen.csv`, `fold_bc_metrics.csv`, `state_slice_bc_metrics.csv`, `selection.json`, `summary.json`.
"""
(OUT_DIR / 'conclusion.md').write_text(conclusion.strip() + '\n')
display(Markdown(conclusion))



## Conclusion

Decision: `short_volshock_three_fold_bc_slice_failed_no_ppo`.

Selected BC config: `plain` with weighted fold accuracy `0.762`, min fold accuracy `0.743`, and gate loss `0.1278`.

Three-fold coverage: required folds `wf_2020_train, wf_2022_train, wf_2023_2024_train`; missing `none`. Short-volshock feature folds: `wf_2020_train, wf_2022_train, wf_2023_2024_train`; missing `none`.

Fold fit:
- wf_2020_train: accuracy 0.743, samples 2157, short_target_samples 382, gate True
- wf_2022_train: accuracy 0.917, samples 253, short_target_samples 133, gate True
- wf_2023_2024_train: accuracy 0.767, samples 755, short_target_samples 335, gate True

Required slice fit:
- drawdown_control_state (universal): min accuracy 0.762, min samples 65, gate True
- is_downtrend_state (universal): min accuracy 0.791, min samples 152, gate True
- release_gate_state (universal): min accuracy 0.472, min samples 36, gate False
- short_volshock_state (short_volshock): min accuracy 0.874, min samples 135, gate True
- short_volshock_target_state (short_volshock): min accuracy 0.872, min samples 133, gate True

Gate note: `bc_screen_gate=False`, but `ppo_screen_allowed_next=False` because full short-volshock feature coverage across all required folds is `True`. `promotion_allowed=false` and `actor_retention_allowed=false` because this is only a BC learnability screen; retained-model promotion still requires official replay, MaxDD < 20%, and return alignment against current best and SOXLBacktest baseline.

Next action: `retune_short_volshock_targets_or_weighting_before_any_ppo`.

Artifacts: `combined_target_contract.csv`, `fold_dataset_audit.csv`, `bc_weight_screen.csv`, `fold_bc_metrics.csv`, `state_slice_bc_metrics.csv`, `selection.json`, `summary.json`.
